# ai04b Task Solutions: SQL Aggregation

**INSTRUCTOR SOLUTIONS — DO NOT DISTRIBUTE**

In [ ]:
import pandas as pd
import sqlite3

nbaConnection = sqlite3.connect('nba_5seasons.db')
print("✅ Connected")

## Q1: Count Players per Team

In [ ]:
playerCountQuery = """
SELECT team_id, COUNT(*) as playerCount
FROM player_season_stats
WHERE season = '2021-22'
GROUP BY team_id
ORDER BY playerCount DESC
"""

playerCounts = pd.read_sql(playerCountQuery, nbaConnection)
display(playerCounts)
print(f"Total teams: {len(playerCounts)}")

### Note
- COUNT(*) counts rows in each group
- GROUP BY team_id creates one group per team
- Returns ~30 teams with player counts

## Q2: Average Points Per Team

In [ ]:
avgPointsQuery = """
SELECT team_id, AVG(pts) as avgPoints, COUNT(*) as gameCount
FROM team_game_stats
WHERE season = '2021-22'
GROUP BY team_id
ORDER BY avgPoints DESC
LIMIT 10
"""

topScoringTeams = pd.read_sql(avgPointsQuery, nbaConnection)
display(topScoringTeams)
print(f"\nTop team averaged {topScoringTeams.iloc[0]['avgPoints']:.1f} PPG")

### Note
- AVG() calculates mean
- Returns 10 highest-scoring teams

## Q3: Total Wins per Team

In [ ]:
winsQuery = """
SELECT team_id, COUNT(*) as totalWins
FROM team_game_stats
WHERE season = '2021-22' AND wl = 'W'
GROUP BY team_id
ORDER BY totalWins DESC
"""

teamWins = pd.read_sql(winsQuery, nbaConnection)
display(teamWins.head(10))
print(f"\nMost wins: {teamWins.iloc[0]['totalWins']} games")

### Note
- WHERE wl = 'W' filters BEFORE grouping
- Only counts win rows

## Q4: Teams with 50+ Wins

In [ ]:
eliteQuery = """
SELECT team_id, COUNT(*) as totalWins
FROM team_game_stats
WHERE season = '2021-22' AND wl = 'W'
GROUP BY team_id
HAVING COUNT(*) >= 50
ORDER BY totalWins DESC
"""

eliteTeams = pd.read_sql(eliteQuery, nbaConnection)
display(eliteTeams)
print(f"\n{len(eliteTeams)} elite teams with 50+ wins")

### Note
- HAVING filters groups AFTER grouping
- KEY DIFFERENCE: WHERE filters rows, HAVING filters groups

## Q5: High-Scoring Games per Team

In [ ]:
highScoringQuery = """
SELECT team_id, COUNT(*) as highScoringGames
FROM team_game_stats
WHERE season = '2021-22' AND pts >= 120
GROUP BY team_id
ORDER BY highScoringGames DESC
LIMIT 10
"""

highScoringCounts = pd.read_sql(highScoringQuery, nbaConnection)
display(highScoringCounts)

### Note
- WHERE pts >= 120 before GROUP BY
- Counts only games with 120+ points

## Q6: Average Points by Position

In [ ]:
# First check if position column exists
structureQuery = "PRAGMA table_info(player_season_stats)"
tableInfo = pd.read_sql(structureQuery, nbaConnection)
columns = tableInfo['name'].tolist()
print(f"Columns: {columns}")
print(f"\nHas 'position' column: {'position' in columns}")

In [ ]:
# If position exists, use it; otherwise group by player_id
if 'position' in columns:
    positionQuery = """
    SELECT position, AVG(pts) as avgPoints, COUNT(*) as playerCount
    FROM player_season_stats
    WHERE season = '2021-22' AND gp >= 40
    GROUP BY position
    ORDER BY avgPoints DESC
    """
    positionStats = pd.read_sql(positionQuery, nbaConnection)
else:
    # Alternative: Group by player and show average
    positionQuery = """
    SELECT AVG(pts) as avgPoints, COUNT(*) as playerCount
    FROM player_season_stats
    WHERE season = '2021-22' AND gp >= 40
    """
    positionStats = pd.read_sql(positionQuery, nbaConnection)

display(positionStats)

### Note
- Checks for position column existence
- gp >= 40 ensures meaningful playing time
- Flexible query design

## Q7: MIN and MAX Points

In [ ]:
minMaxQuery = """
SELECT team_id, MAX(pts) as maxPoints, MIN(pts) as minPoints, AVG(pts) as avgPoints
FROM team_game_stats
WHERE season = '2021-22'
GROUP BY team_id
ORDER BY maxPoints DESC
LIMIT 5
"""

minMaxStats = pd.read_sql(minMaxQuery, nbaConnection)
display(minMaxStats)

### Note
- MAX() and MIN() find extremes
- Useful for range analysis
- Shows scoring variability

In [ ]:
nbaConnection.close()
print("✅ Closed")

---

## Key Patterns

### Aggregation Pattern
```sql
SELECT groupColumn, AGGREGATE(dataColumn) as alias
FROM table
WHERE condition
GROUP BY groupColumn
HAVING groupCondition
ORDER BY alias
```

### WHERE vs HAVING
- **WHERE:** Filters rows BEFORE grouping
- **HAVING:** Filters groups AFTER grouping

### Aggregate Functions
- COUNT(*) - rows in group
- SUM() - total
- AVG() - average
- MIN() - minimum
- MAX() - maximum

These are critical for ML feature engineering!